# Week 08. FastAPI 핵심: Path, Query, Request Body

8주차는 FastAPI 공식 문서의 기본 흐름인 Path Parameter, Query Parameter, Request Body를 다룹니다.
현재 환경에 FastAPI가 없어도 실행되도록, 같은 개념을 순수 함수와 Pydantic으로 재현합니다.


## 제출자 정보

- 이름: 이성민
- 학과: 소프트웨어융합과
- 학년: 2학년
- 학번: 2151050


## 1. 실행 환경 확인

FastAPI가 설치되어 있으면 실제 앱을 만들 수 있고, 없으면 개념 재현 코드만 실행합니다.
중요한 점은 노트북 전체가 오류 없이 끝나는 것입니다.


In [1]:
try:
    import fastapi  # noqa: F401
    FASTAPI_AVAILABLE = True
except ImportError:
    FASTAPI_AVAILABLE = False

print("FastAPI installed:", FASTAPI_AVAILABLE)


FastAPI installed: False


## 2. Path Parameter와 Query Parameter

Path Parameter는 URL 경로 자체에 들어가는 값이고, Query Parameter는 `?key=value` 형태로 붙는 선택 조건입니다.


In [2]:
fake_items = {
    1: {"name": "Notebook", "price": 2500},
    2: {"name": "Pen", "price": 800},
    3: {"name": "Backpack", "price": 42000},
}


def read_item(item_id: int, q: str | None = None):
    item = fake_items.get(item_id)
    if item is None:
        return {"status": 404, "detail": "item not found"}
    result = {"status": 200, "item_id": item_id, "item": item}
    if q:
        result["query"] = q
    return result


print(read_item(1))
print(read_item(2, q="school"))
print(read_item(999))


{'status': 200, 'item_id': 1, 'item': {'name': 'Notebook', 'price': 2500}}
{'status': 200, 'item_id': 2, 'item': {'name': 'Pen', 'price': 800}, 'query': 'school'}
{'status': 404, 'detail': 'item not found'}


## 3. Request Body 검증

POST/PUT 요청의 본문은 Pydantic 모델로 검증할 수 있습니다.
잘못된 가격이나 빈 이름은 모델 생성 단계에서 걸러집니다.


In [3]:
from pydantic import BaseModel, Field, ValidationError


class ItemBody(BaseModel):
    name: str = Field(min_length=1)
    price: float = Field(gt=0)
    tags: list[str] = Field(default_factory=list)


def create_item(item_id: int, body: dict):
    try:
        item = ItemBody(**body)
    except ValidationError as error:
        return {"status": 422, "errors": len(error.errors())}
    return {"status": 201, "item_id": item_id, "item": item.model_dump() if hasattr(item, "model_dump") else item.dict()}


print(create_item(4, {"name": "Marker", "price": 1500, "tags": ["stationery"]}))
print(create_item(5, {"name": "", "price": -1}))


{'status': 201, 'item_id': 4, 'item': {'name': 'Marker', 'price': 1500.0, 'tags': ['stationery']}}
{'status': 422, 'errors': 2}


## 정리

- Path Parameter는 필수 식별자에 적합합니다.
- Query Parameter는 검색, 필터, 정렬 같은 선택 조건에 적합합니다.
- Request Body는 구조가 있는 입력을 받을 때 사용합니다.
